In [1]:
import sys
sys.path.insert(0, '.')

from db import (
    get_boms,
    get_bom_components,
    batch,
)

In [2]:
boms = get_boms()
print(f"{len(boms)} BOMs")
boms[:3]

149 BOMs


[{'BOMId': 48,
  'ProducedSKU': 'FG-amazon-b0002wrqy4',
  'CompanyName': 'ALL ONE'},
 {'BOMId': 57,
  'ProducedSKU': 'FG-amazon-b007vwbm2u',
  'CompanyName': 'Vitacost'},
 {'BOMId': 49,
  'ProducedSKU': 'FG-amazon-b012t9ga6c',
  'CompanyName': 'Optimum Nutrition'}]

In [3]:
# ── Supplier batching for a product ──────────────────────────────────────────
PRODUCT_SKU = boms[0]['ProducedSKU']   # change to any finished-good SKU

result = batch(PRODUCT_SKU)

if not result['assignments'] and not result['uncovered']:
    print(f"No BOM found for {PRODUCT_SKU}")
else:
    bom_lookup = {b['ProducedSKU']: b['BOMId'] for b in boms}
    bom_id = bom_lookup.get(PRODUCT_SKU)
    components = get_bom_components(bom_id)

    # old_supplier = owning company on the component; price/quality not in DB
    old_supplier_map = {c['ConsumedSKU']: c['CompanyName'] for c in components}

    rows = []
    for c in components:
        sku = c['ConsumedSKU']
        rows.append({
            'ingredient': sku,
            'old_supplier': old_supplier_map.get(sku, 'N/A'),
            'new_supplier': result['assignments'].get(sku, '— uncovered —'),
            'old_price':   'N/A',
            'new_price':   'N/A',
            'old_quality': 'N/A',
            'new_quality': 'N/A',
        })
    rows.sort(key=lambda r: r['ingredient'])

    print(f"Product : {PRODUCT_SKU}")
    print(f"Suppliers after batching: {len(result['suppliers'])}  |  "
          f"Uncovered: {len(result['uncovered'])}\n")

    ing_w = 48
    sup_w = 22
    prc_w = 10
    qlt_w = 12
    header = (
        f"{'Ingredient':<{ing_w}} "
        f"{'Old Supplier':<{sup_w}} {'New Supplier':<{sup_w}} "
        f"{'Old Price':<{prc_w}} {'New Price':<{prc_w}} "
        f"{'Old Quality':<{qlt_w}} {'New Quality':<{qlt_w}}"
    )
    print(header)
    print('-' * len(header))
    for r in rows:
        print(
            f"{r['ingredient']:<{ing_w}} "
            f"{r['old_supplier']:<{sup_w}} {r['new_supplier']:<{sup_w}} "
            f"{r['old_price']:<{prc_w}} {r['new_price']:<{prc_w}} "
            f"{r['old_quality']:<{qlt_w}} {r['new_quality']:<{qlt_w}}"
        )

Product : FG-amazon-b0002wrqy4
Suppliers after batching: 8  |  Uncovered: 0

Ingredient                                       Old Supplier           New Supplier           Old Price  New Price  Old Quality  New Quality 
----------------------------------------------------------------------------------------------------------------------------------------------
RM-C2-ascorbic-acid-4823fabf                     ALL ONE                Prinova USA            N/A        N/A        N/A          N/A         
RM-C2-beta-carotene-bb90cd60                     ALL ONE                AIDP                   N/A        N/A        N/A          N/A         
RM-C2-choline-bitartrate-296d4da4                ALL ONE                Prinova USA            N/A        N/A        N/A          N/A         
RM-C2-cyanocobalamin-c2d9669b                    ALL ONE                Prinova USA            N/A        N/A        N/A          N/A         
RM-C2-d-alpha-tocopheryl-succinate-532e67fd      ALL ONE         